In [1]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
DATA_DIR = "/content/drive/MyDrive/sentiment labelled sentences"

In [4]:
import pandas as pd

amazon = pd.read_csv(
    DATA_DIR + "/amazon_cells_labelled.txt",
    sep="\t",
    header=None,
    names=["sentence", "label"]
)

imdb = pd.read_csv(
    DATA_DIR + "/imdb_labelled.txt",
    sep="\t",
    header=None,
    names=["sentence", "label"]
)

yelp = pd.read_csv(
    DATA_DIR + "/yelp_labelled.txt",
    sep="\t",
    header=None,
    names=["sentence", "label"]
)

In [5]:
df = pd.concat(
    [amazon, imdb, yelp],
    ignore_index=True
)

In [6]:
print(df.shape)
print(df.head())
print(df["label"].value_counts())

(2748, 2)
                                            sentence  label
0  So there is no way for me to plug it in here i...      0
1                        Good case, Excellent value.      1
2                             Great for the jawbone.      1
3  Tied to charger for conversations lasting more...      0
4                                  The mic is great.      1
label
1    1386
0    1362
Name: count, dtype: int64


In [7]:
def clean_text(text):
    text = text.lower()
    text = re.sub(
        r"[^a-zA-Z0-9\s']",
        "",
        text
    )
    return text
df["sentence"] = df["sentence"].apply(
    clean_text
)

In [8]:
X = df["sentence"]
y = df["label"]


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))




Training samples: 2198
Testing samples: 550


In [10]:
tokenizer = Tokenizer(
    num_words=5000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(
    X_train
)


X_train_sequences = tokenizer.texts_to_sequences(
    X_train
)

X_test_sequences = tokenizer.texts_to_sequences(
    X_test
)

print("\nSentence:")
print(X_train.iloc[0])

print("\nInteger encoded:")
print(X_train_sequences[0])


Sentence:
i do not know if this was emilio estevez's directorial debut but the pacing the interplay and development of the characters as well as some clever camera work surrounding the character estevez plays all suggest a natural eye  

Integer encoded:
[5, 109, 15, 162, 47, 9, 11, 1199, 1882, 1200, 1883, 24, 2, 1201, 2, 1202, 3, 1203, 10, 2, 140, 29, 54, 29, 100, 717, 191, 110, 1204, 2, 213, 1884, 882, 34, 1205, 4, 1206, 1885]


In [11]:
max_length = 40


X_train_pad = pad_sequences(
    X_train_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

print("\nAfter padding:")
print(X_train_pad[0])



After padding:
[   5  109   15  162   47    9   11 1199 1882 1200 1883   24    2 1201
    2 1202    3 1203   10    2  140   29   54   29  100  717  191  110
 1204    2  213 1884  882   34 1205    4 1206 1885    0    0]


In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

model = Sequential([

    Embedding(
        input_dim=5000,
        output_dim=64,
        mask_zero=True
    ),

    SimpleRNN(
        units=64
    ),

    Dropout(0.3),

    Dense(
        units=1,
        activation="sigmoid"
    )

])

In [13]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)


Epoch 1/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.5660 - loss: 0.6874 - val_accuracy: 0.6114 - val_loss: 0.6784
Epoch 2/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.8635 - loss: 0.4748 - val_accuracy: 0.6864 - val_loss: 0.6354
Epoch 3/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.9562 - loss: 0.1780 - val_accuracy: 0.6977 - val_loss: 0.6606
Epoch 4/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.9937 - loss: 0.0615 - val_accuracy: 0.7523 - val_loss: 0.6564
Epoch 5/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9994 - loss: 0.0294 - val_accuracy: 0.7455 - val_loss: 0.7108
Epoch 6/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9994 - loss: 0.0170 - val_accuracy: 0.7364 - val_loss: 0.7552
Epoch 7/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 1.0000 - loss: 0.0114 - val_accuracy: 0.7523 - val_loss: 0.7820
Epoch 8/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 1.0000 - loss: 0.0080 - val_accuracy: 0.7568 - v

In [15]:
loss, accuracy = model.evaluate(
    X_test_pad,
    y_test
)

print("Test Accuracy:", accuracy)


18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7436 - loss: 1.1939
Test Accuracy: 0.7436363697052002


In [16]:
def predict_sentiment(sentence):

    sentence = clean_text(
        sentence
    )
    sequence = tokenizer.texts_to_sequences(
        [sentence]
    )
    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post",
        truncating="post"
    )
    prediction = model.predict(
        padded,
        verbose=0
    )[0][0]

    if prediction >= 0.5:

        print("Sentiment: Positive")

    else:

        print("Sentiment: Negative")


    print(
        "Score:",
        prediction
    )


In [17]:
predict_sentiment(
    "I really loved this movie"
)

predict_sentiment(
    "The food was terrible"
)

Sentiment: Positive
Score: 0.9996189
Sentiment: Negative
Score: 0.00095237774
